# Figure 1 — MIL vs. standard supervised learning (schematic)

Purely illustrative, sized for one PRL column. Values are shaded in blues (kept distinct from the viridis of Figs. 2-4);
orange marks the arg-max instance. Bags are written as sets,
$\{x_1, x_2, \dots\}$, and the label is read from the first feature ($\beta=e_1$).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Rectangle

from extremal_mil import plotting

plt.rcParams.update(plotting.RCPARAMS)
plt.rcParams.update({"font.size": 8})

get_ipython().run_line_magic('config', "InlineBackend.figure_format = 'svg'")

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ACCENT = "#d95f02"   # arg-max instance
INK = "0.3"
# blues, not the viridis of Figs. 2-4, where color encodes u_0
CMAP = mcolors.LinearSegmentedColormap.from_list(
    "soft_blues", plt.cm.Blues(np.linspace(0.15, 0.85, 256)))
NORM = mcolors.Normalize(vmin=-2.0, vmax=2.0)

# Layout in data units with equal aspect; the figure height follows from the
# limits so that it is always one column (3.4 in) wide.
COL_W = 3.4
N_ROWS, N_FEAT, N_INST = 3, 4, 4
CW, CH = 0.55, 0.42      # feature cell width, height
INSET = 0.07             # cells sit this far inside the instance outline
VEC_H = N_FEAT * CH
ROW_GAP = 1.1
R = 0.38                 # label circle radius
ARROW = 2.2
GAP = 0.45               # between instances in a bag
BAG_W = 0.9 + N_INST * CW + (N_INST - 1) * GAP   # brace to brace

## Panel helpers

In [ ]:
def row_y(i):
    return -i * (VEC_H + ROW_GAP)


def draw_vector(ax, x0, y0, v, selected=False):
    # one instance, feature 1 on top, cells inset from a rounded outline
    ch = (VEC_H - 2 * INSET) / len(v)
    for f, val in enumerate(v):
        ax.add_patch(Rectangle((x0 + INSET, y0 + VEC_H - INSET - (f + 1) * ch),
                               CW - 2 * INSET, ch, fc=CMAP(NORM(val)), ec="white", lw=0.5))
    ax.add_patch(FancyBboxPatch((x0, y0), CW, VEC_H, boxstyle="round,pad=0,rounding_size=0.1",
                                fill=False, ec=ACCENT if selected else INK,
                                lw=1.4 if selected else 0.5))


def draw_brace(ax, x, y0, h, side):
    """Thin curly brace over [y0, y0 + h]; side=-1 draws {, side=+1 draws }."""
    t = np.linspace(0, 1, 200)
    w = 0.22 * side
    cusp = np.exp(-((t - 0.5) / 0.035) ** 2)
    tips = np.exp(-(np.minimum(t, 1 - t) / 0.05) ** 2)
    # the cusp points away from the contents, the tips curl toward them
    ax.plot(x + w * cusp - 0.6 * w * tips, y0 + h * t, color=INK, lw=0.7)


def draw_readout(ax, x0, y, value):
    ax.add_patch(FancyArrowPatch((x0, y), (x0 + ARROW, y), arrowstyle="-|>",
                                 mutation_scale=6, color=INK, lw=0.6, shrinkA=0, shrinkB=0))
    cx = x0 + ARROW + 0.2 + R
    ax.add_patch(Circle((cx, y), R, fc=CMAP(NORM(value)), ec=INK, lw=0.5))
    return cx


def panel_supervised(ax, x0, rng):
    """Returns (input centre, arrow start, label centre)."""
    a0 = x0 + CW + 0.5
    for i in range(N_ROWS):
        y0 = row_y(i)
        v = rng.standard_normal(N_FEAT)
        draw_vector(ax, x0, y0, v)
        cx = draw_readout(ax, a0, y0 + VEC_H / 2, v[0])
    return x0 + CW / 2, a0, cx


def panel_mil(ax, x0, rng):
    """Returns (input centre, arrow start, label centre)."""
    a0 = x0 + BAG_W + 0.35
    for i in range(N_ROWS):
        y0 = row_y(i)
        data = rng.standard_normal((N_INST, N_FEAT))
        # w^k = max_j <x_j, beta> with beta = e_1: the largest first feature
        sel = int(np.argmax(data[:, 0]))
        # a bag is a set, written as one: {x_1, x_2, ...}
        xs = x0 + 0.45 + np.arange(N_INST) * (CW + GAP)
        for j in range(N_INST):
            draw_vector(ax, xs[j], y0, data[j], selected=(j == sel))
            if j < N_INST - 1:
                ax.text(xs[j] + CW + GAP / 2, y0, ",", ha="center", va="baseline",
                        fontsize=10, color=INK)
        draw_brace(ax, x0 + 0.15, y0 - 0.2, VEC_H + 0.4, -1)
        draw_brace(ax, x0 + BAG_W - 0.15, y0 - 0.2, VEC_H + 0.4, +1)
        cx = draw_readout(ax, a0, y0 + VEC_H / 2, data[sel, 0])
    return x0 + BAG_W / 2, a0, cx

## Assemble and save

In [ ]:
fig, ax = plt.subplots()
fig.subplots_adjust(0, 0, 1, 1)
ax.set_aspect("equal")
ax.axis("off")

mid_a, a_a, cx_a = panel_supervised(ax, 0.0, np.random.default_rng(0))
x1 = cx_a + R + 1.3
mid_b, a_b, cx_b = panel_mil(ax, x1, np.random.default_rng(5))

top = VEC_H + 0.5
for x, label in [(mid_a, "$x_i$"), (cx_a, "$y_i$"), (mid_b, "$X^k$"), (cx_b, "$w^k$")]:
    ax.text(x, top, label, ha="center", va="bottom", fontsize=9)
for a0, rule in [(a_a, r"$x_i^{\top}\beta$"), (a_b, r"$\mathrm{max}_j\, x_j^{\top}\beta$")]:
    ax.text(a0 + ARROW / 2, VEC_H / 2 + 0.12, rule, ha="center", va="bottom", color=INK)
ax.text((cx_a + R) / 2, top + 1.0, "Standard\nsupervised learning", ha="center", va="bottom", fontsize=9, linespacing=1.1)
ax.text((x1 + cx_b + R) / 2, top + 1.0, "Multiple\ninstance learning", ha="center", va="bottom",
        fontsize=9, linespacing=1.1)

xlim, ylim = (-0.5, cx_b + R + 0.3), (row_y(N_ROWS - 1) - 0.4, top + 2.3)
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
fig.set_size_inches(COL_W, COL_W * (ylim[1] - ylim[0]) / (xlim[1] - xlim[0]))

fig.savefig(FIG_DIR / "fig1_intro.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()